In [1]:
import os
import sys

CURRENT_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, os.pardir))
LLM_DIR = os.path.join(PROJECT_ROOT, "llm_results")  # ou "llm_results/arara" se for o seu caso
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [ ]:

database = 'neo4j'
query_type = 'condition'
dataset_type = 'ExlicitQuery'  # ou 'ExplicitQuery' dependendo do seu caso
groundtruths = os.path.join(PROJECT_ROOT, "dataset", "movie", f"{dataset_type}.json")

import os, json, itertools

def is_candidate(file_name: str, eval_type: str) -> bool:
    return (
        eval_type in file_name
        and ("output" in file_name or "prediction" in file_name)
        and file_name.endswith(".jsonl")
    )

def validate_predictions_file(path: str):
    if not os.path.isfile(path):
        return False, "arquivo não existe"
    if os.path.getsize(path) == 0:
        return False, "arquivo vazio (0 bytes)"
    # lê a primeira linha não-vazia
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception as e:
                return False, f"JSON inválido na 1ª linha não-vazia: {e}"
            if not isinstance(obj, dict):
                return False, "1ª linha não é um objeto JSON"
            if "id" not in obj or "response" not in obj:
                return False, "faltam chaves obrigatórias: 'id' e/ou 'response'"
            return True, "ok"
    return False, "somente linhas vazias"

# --- use ---
# garanta que dataset_type exista (ex.: "ImplicitQuery" ou "ExplicitQuery")
# dataset_type = "ImplicitQuery"
eval_type = f"movie-{dataset_type}"

avaliados = 0
pulados = 0

for root, dirs, files in os.walk(LLM_DIR):
    print("root=", root)
    for file in files:
        print("file=", file)
        if not is_candidate(file, eval_type):
            continue

        predictions = os.path.join(root, file)
        ok, msg = validate_predictions_file(predictions)
        if not ok:
            print(f"🟡 Pulando: {predictions} — {msg}")
            pulados += 1
            continue

        print(f"✅ Avaliando: {predictions}")
        scrit_name = (
            f'eval_movie.py --database {database} '
            f'--query_type {query_type} '
            f'--groundtruths "{groundtruths}" '
            f'--predictions "{predictions}"'
        )
        get_ipython().run_line_magic('run', scrit_name)
        print('---------------------------------------------')
        avaliados += 1

print(f"\nResumo: {avaliados} avaliados, {pulados} pulados.")



root= /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results
file= .DS_Store
file= grpo-predictions.jsonl
file= grpo-predictions_output.jsonl
root= /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results/gpt-4o-k
file= movie-ExplicitQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-MisinformedQuery_gpt-4o-historyTrue-k=5-OpenaiBatchFile-0.jsonl
file= movie-ExplicitQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
✅ Avaliando: /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results/gpt-4o-k/movie-ExplicitQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl


INFO:root:Successfully connected to the Neo4j database.
Evaluating predictions: 0it [00:00, ?it/s]
INFO:root: Condition Num: 1.6583950370222134
INFO:root: Ftr: 0.0020012007204322593
INFO:root: Recall: 0.43612437783422575
INFO:root: Precision: 0.23281969181508907
INFO:root: Ndcg: 0.4055948121329651
INFO:root: Satisfied Ratio: 0.6569118252987227
INFO:root: Existence In Kg Ratio: 0.7426455873524115
INFO:root:Successfully connected to the Neo4j database.


---------------------------------------------
file= movie-ItemBasedQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-ExplicitQuery_gpt-4o-historyTrue-k=5-OpenaiBatchFile-0_output.jsonl
✅ Avaliando: /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results/gpt-4o-k/movie-ExplicitQuery_gpt-4o-historyTrue-k=5-OpenaiBatchFile-0_output.jsonl


Evaluating predictions: 0it [00:00, ?it/s]
INFO:root: Condition Num: 1.6583950370222134
INFO:root: Ftr: 0.003201921152691615
INFO:root: Recall: 0.45350216381317243
INFO:root: Precision: 0.24274564738843307
INFO:root: Ndcg: 0.4179855297608083
INFO:root: Satisfied Ratio: 0.6486063305662415
INFO:root: Existence In Kg Ratio: 0.8089653792275365
INFO:root:Successfully connected to the Neo4j database.


---------------------------------------------
file= movie-ExplicitQuery_gpt-4o-historyTrue-k=5-OpenaiBatchFile-0.jsonl
file= movie-ImplicitQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-MisinformedQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-ItemBasedQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-ImplicitQuery_gpt-4o-historyTrue-k=5-OpenaiBatchFile-0.jsonl
file= movie-UserBasedQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-ImplicitQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-ImplicitQuery_gpt-4o-historyTrue-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-MisinformedQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-MisinformedQuery_gpt-4o-historyTrue-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-UserBasedQuery_gpt-4o-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
root= /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/ll

...

---------------------------------------------
root= /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results/claude-3-5-sonnet-20241022-k
file= movie-ExplicitQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-ExplicitQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
✅ Avaliando: /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results/claude-3-5-sonnet-20241022-k/movie-ExplicitQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl


Evaluating predictions: 0it [00:00, ?it/s]
INFO:root: Condition Num: 1.6583950370222134
INFO:root: Ftr: 0.0016009605763458075
INFO:root: Recall: 0.45992012676531907
INFO:root: Precision: 0.24764858915349208
INFO:root: Ndcg: 0.4299944920279103
INFO:root: Satisfied Ratio: 0.61793661250183
INFO:root: Existence In Kg Ratio: 0.4713428056834101


---------------------------------------------
file= movie-ItemBasedQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-ImplicitQuery_claude-3-5-sonnet-20241022-historyTrue-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-MisinformedQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-UserBasedQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-ExplicitQuery_claude-3-5-sonnet-20241022-historyTrue-k=5-OpenaiBatchFile-0.jsonl
file= movie-MisinformedQuery_claude-3-5-sonnet-20241022-historyTrue-k=5-OpenaiBatchFile-0.jsonl
file= movie-ImplicitQuery_claude-3-5-sonnet-20241022-historyTrue-k=5-OpenaiBatchFile-0.jsonl
file= movie-ExplicitQuery_claude-3-5-sonnet-20241022-historyTrue-k=5-OpenaiBatchFile-0_output.jsonl
✅ Avaliando: /Users/fillipesantos/Documents/projects/arara_experiments/datasets/recassistbench/llm_results/claude-3-5-sonnet-20241022-k/movie-ExplicitQuery_claude-3-5-sonnet-2024

INFO:root:Successfully connected to the Neo4j database.
Evaluating predictions: 0it [00:00, ?it/s]
INFO:root: Condition Num: 1.6583950370222134
INFO:root: Ftr: 0.0
INFO:root: Recall: 0.45809214871045933
INFO:root: Precision: 0.24802881729037424
INFO:root: Ndcg: 0.42320114978558365
INFO:root: Satisfied Ratio: 0.625539255091104
INFO:root: Existence In Kg Ratio: 0.8514708825295177


---------------------------------------------
file= movie-ImplicitQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-MisinformedQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-ImplicitQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-MisinformedQuery_claude-3-5-sonnet-20241022-historyTrue-k=5-OpenaiBatchFile-0_output.jsonl
file= movie-ItemBasedQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0.jsonl
file= movie-UserBasedQuery_claude-3-5-sonnet-20241022-historyFalse-k=5-OpenaiBatchFile-0_output.jsonl

Resumo: 20 avaliados, 0 pulados.
